## DATA-DRIVEN OPTIMIZATION MODEL FOR USED VEHICLE PROCUREMENT IN REMANUFACTURING
-----------------------
### Author: 
### Shuvendu Pritam Das
- **Email**: 23mt0389@iitism.ac.in
- **LinkedIn**: [Shuvendu Pritam Das](https://linkedin.com/in/shuvendupritamdas)
- **GitHub**: [SPritamDas](https://github.com/SPritamDas)

In [ ]:
# Importing Libraries for Data Handling
import numpy as np
import pandas as pd

# Configuring pandas display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Importing Libraries for Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Importing Standard Libraries
import os
import time
import sys
import datetime

# Suppressing Warnings
import warnings
warnings.filterwarnings('ignore')

#### Merging

In [ ]:
def merge_csv_files(input_folder: str, output_folder: str, output_filename: str):
    os.makedirs(output_folder, exist_ok=True)
    csv_files = [file for file in os.listdir(input_folder) if file.endswith('.csv')]
    data_frames = []
    
    for file in csv_files:
        file_path = os.path.join(input_folder, file)
        df = pd.read_csv(file_path)
        data_frames.append(df)

    merged_ = pd.concat(data_frames, ignore_index=True)
    output_file_path = os.path.join(output_folder, output_filename)
    merged_.to_csv(output_file_path, index=False)
    print(f"All files have been merged into {output_file_path}")

# Load and merge data
merge_csv_files(r"C:\Users\shuve\Desktop\Thesis\Mythesis\Allscarped data", r"C:\Users\shuve\Desktop\Thesis\Mythesis\Codes\Final Data", "merged_file.csv")
df = pd.read_csv(r"C:\Users\shuve\Desktop\Thesis\Mythesis\Codes\Final Data\merged_file.csv")

In [ ]:
df.info()

#### Feature Selection

In [ ]:
# Creating list of important features
feature_list = ['parameters/0/formatted_value', 'parameters/1/formatted_value', 
                'parameters/4/formatted_value', 'parameters/5/formatted_value', 
                'parameters/6/value', 'parameters/7/value', 'price/raw']

# Selecting features and renaming columns
df = df[feature_list].rename(columns={
    'parameters/0/formatted_value': 'brand',
    'parameters/1/formatted_value': 'model',
    'parameters/4/formatted_value': 'fuel_type',
    'parameters/5/formatted_value': 'transmission_type',
    'price/raw': 'procurement_cost',
    'parameters/7/value': 'prev_owner_count',
    'parameters/6/value': 'mileage'
})

# Converting text to lowercase
text_columns = ['brand', 'model', 'fuel_type', 'transmission_type']
for col in text_columns:
    df[col] = df[col].str.lower()

#### Filtering

In [ ]:
# Filter relevant entries
valid_models = ['wagonr', 'swift', 'breeza', 'kwid', 'baleno', 'creta', 
                'i10', 'i20', 'scorpio', 'celerio', 'ertiga', 'duster', 
                'alto800', 'dzire', 'ecosport']

valid_fuel_types = ['petrol', 'cng & hybrids', 'diesel', 'lpg']
valid_transmission = ['manual', 'automatic']
valid_owner_counts = list(range(1, 7))

df = df[df['model'].isin(valid_models)]
df = df[(df['fuel_type'].isin(valid_fuel_types)) | df['fuel_type'].isnull()]
df = df[(df['transmission_type'].isin(valid_transmission)) | df['transmission_type'].isnull()]
df = df[(df['prev_owner_count'].isin(valid_owner_counts)) | df['prev_owner_count'].isnull()]

In [ ]:
# Car Type Categorization
def categorize_car_type(model):
    if model in ['breeza', 'creta', 'scorpio', 'duster', 'ecosport']:
        return 'suv'
    elif model == 'ertiga':
        return 'muv'
    elif model in ['swift', 'baleno', 'alto800', 'wagonr', 'celerio', 'i10', 'kwid', 'i20']:
        return 'hatchback'
    elif model == 'dzire':
        return 'sedan'
    return 'not_defined'

# Price Segment Categorization
def categorize_price_segment(model):
    segments = {
        'wagonr': '<5 lakh', 'swift': '5-10 lakh', 'breeza': '>10 lakh',
        'kwid': '<5 lakh', 'baleno': '5-10 lakh', 'creta': '>10 lakh',
        'i10': '<5 lakh', 'i20': '5-10 lakh', 'scorpio': '>10 lakh',
        'celerio': '<5 lakh', 'ertiga': '5-10 lakh', 'duster': '>10 lakh',
        'alto800': '<5 lakh', 'dzire': '5-10 lakh', 'ecosport': '>10 lakh'
    }
    return segments.get(model, 'Unknown')

df['car_type'] = df['model'].apply(categorize_car_type)
df['price_segment'] = df['model'].apply(categorize_price_segment)

#### Duplicate

In [ ]:
# Check and remove duplicates
print('Number of duplicated rows before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Number of duplicated rows after:', df.duplicated().sum())

#### Feature Generation

In [ ]:
# Define brand new prices for each model
model_to_actual_price = {
    'wagonr': 555000,
    'swift': 649000,
    'breeza': 834000,
    'kwid': 470000,
    'baleno': 660000,
    'creta': 1100000,
    'i10': 592000,
    'i20': 700000,
    'scorpio': 1362000,
    'celerio': 536000,
    'ertiga': 870000,
    'duster': 100000,
    'alto800': 420000,
    'dzire': 700000,
    'ecosport': 900000
}

# Add brand new prices to dataframe
df['brand_new_on_road_price'] = df['model'].map(model_to_actual_price)

#### Focus on `5-10 lakh` price segment

In [ ]:
df = df[df['price_segment'] == '5-10 lakh']

#### Null Values    

In [ ]:
# Handle missing values
print('\nNull values before cleaning:')
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df)) * 100
null_summary = pd.DataFrame({'Null Count': null_counts, 'Null Percentage': null_percentages})
print(null_summary)

In [ ]:

# Calculate value counts and convert to percentages
value_counts = df['prev_owner_count'].value_counts().sort_index()
percentages = (value_counts / value_counts.sum()) * 100

# Create a horizontal bar chart
plt.figure(figsize=(8, 4))  # Adjusted for a horizontal layout
colors = plt.cm.Paired(range(len(percentages)))  # Professional color palette
bars = plt.barh(
    [f'{count} Owner(s)' for count in percentages.index],
    percentages,
    color=colors,
    edgecolor='white',
    linewidth=1.2
)

# Add percentage values on the bars
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.5,  # Slightly right of the bar
        bar.get_y() + bar.get_height() / 2,
        f'{width:.1f}%',
        ha='left',
        va='center',
        fontsize=10,
        weight='bold'
    )

# Customize the chart
plt.title('Distribution of Previous Owner Count', fontsize=12, pad=10, weight='bold')
plt.xlabel('Percentage (%)', fontsize=10)
plt.ylabel('Previous Owners', fontsize=10)
plt.xlim(0, 70)  # Set x-axis limit to accommodate values
plt.grid(axis='x', linestyle='--', alpha=0.7)  # Add subtle grid for readability

# Adjust layout to prevent clipping
plt.tight_layout()

# Save and display the chart
plt.savefig('bar_chart.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Fill missing prev_owner_count with mode
prev_owner_count_mode = df['prev_owner_count'].mode()[0]
df['prev_owner_count'] = df['prev_owner_count'].fillna(prev_owner_count_mode)
df['prev_owner_count'] = df['prev_owner_count'].astype('object')

# Check null values after filling prev_owner_count
print('\nNull values after filling prev_owner_count:')
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df)) * 100
null_summary = pd.DataFrame({'Null Count': null_counts, 'Null Percentage': null_percentages})
print(null_summary)

# Drop remaining null values
df = df.dropna()
df['mileage'] = df['mileage'].astype(int)

print('\nDataset shape after cleaning:', df.shape)

In [ ]:
df = df.dropna()
print('\nDataset shape after cleaning:', df.isnull().sum())
print('\nDataset shape after cleaning:', df.shape)


### `Outlier` Handling

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create box plot
plt.figure(figsize=(8, 6))
sns.boxplot(x=df['mileage'])
plt.title('Box Plot of Mileage')
plt.xlabel('Mileage')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression

# Assuming df is your DataFrame with 'mileage' (x) and 'procurement_cost' (y)

# Method 1: IQR on residuals from linear regression
def remove_residual_outliers(df, x_col, y_col):
    # Fit a linear regression model
    X = df[[x_col]].values
    y = df[y_col].values
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    
    # Calculate residuals
    residuals = y - y_pred
    
    # Apply IQR on residuals
    Q1 = np.percentile(residuals, 25)
    Q3 = np.percentile(residuals, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Keep points where residuals are within bounds
    mask = (residuals >= lower_bound) & (residuals <= upper_bound)
    return df[mask]

# Method 2: Isolation Forest on bivariate data
def remove_if_outliers(df, x_col, y_col, contamination=0.01):
    # Prepare data for Isolation Forest
    X = df[[x_col, y_col]].values
    # Fit Isolation Forest
    iso_forest = IsolationForest(contamination=contamination, random_state=42)
    y_pred = iso_forest.fit_predict(X)
    # Keep only non-outliers (where prediction is 1)
    return df[y_pred == 1]

# Apply both methods
df_residual = remove_residual_outliers(df, 'mileage', 'procurement_cost')
df_if = remove_if_outliers(df, 'mileage', 'procurement_cost')

# Create visualizations
plt.figure(figsize=(15, 10))

# 1. Original Scatter Plot
plt.subplot(2, 2, 1)
sns.scatterplot(data=df, x='mileage', y='procurement_cost')
plt.title('Original Mileage vs Procurement Cost')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')

# 2. Residuals Boxplot (to show outliers in residuals)
plt.subplot(2, 2, 2)
X = df[['mileage']].values
y = df['procurement_cost'].values
model = LinearRegression()
model.fit(X, y)
residuals = y - model.predict(X)
sns.boxplot(x=residuals)
plt.title('Residuals Boxplot (Linear Regression)')
plt.xlabel('Residuals')

# 3. Residual Cleaned Scatter Plot
plt.subplot(2, 2, 3)
sns.scatterplot(data=df_residual, x='mileage', y='procurement_cost')
plt.title('Mileage vs Procurement Cost (Residual IQR Method)')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')

# 4. Isolation Forest Cleaned Scatter Plot
plt.subplot(2, 2, 4)
sns.scatterplot(data=df_if, x='mileage', y='procurement_cost')
plt.title('Mileage vs Procurement Cost (Isolation Forest)')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')

plt.tight_layout()
plt.show()

# Print statistics
print("Original dataset size:", len(df))
print("After Residual IQR method:", len(df_residual))
print("After Isolation Forest:", len(df_if))

# Optional: Scatter plot to compare all points
plt.figure(figsize=(10, 6))
plt.scatter(df['mileage'], df['procurement_cost'], c='blue', alpha=0.5, label='All Data')
plt.scatter(df_residual['mileage'], df_residual['procurement_cost'], c='green', alpha=0.5, label='Residual IQR Cleaned')
plt.scatter(df_if['mileage'], df_if['procurement_cost'], c='red', alpha=0.5, label='Isolation Forest Cleaned')
plt.title('Mileage vs Procurement Cost Comparison')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()
plt.show()

In [ ]:
df = df_if.copy()

In [ ]:
df = df.sort_values(by='mileage', ascending=True).copy()

### `Uni-Variate` Analysis

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def univariate_analysis(df):
    # Convert mileage to float
    df['mileage'] = pd.to_numeric(df['mileage'], errors='coerce')

    # Set style for better visualization
    sns.set_style("whitegrid")

    # Specified numerical and categorical columns
    numerical_cols = ['procurement_cost', 'mileage']
    categorical_cols = ['brand', 'model', 'fuel_type', 'transmission_type', 
                       'prev_owner_count', 'car_type', 'price_segment']

    # Combined plot for numerical columns
    plt.figure(figsize=(12, 6))
    for i, col in enumerate(numerical_cols, 1):
        plt.subplot(1, 2, i)
        sns.kdeplot(data=df, x=col, fill=True, label=col, color='blue' if col == 'procurement_cost' else 'red')
        plt.title(col, fontsize=12)
        plt.xlabel('')
        plt.ylabel('Density' if i == 1 else '', fontsize=10)
    plt.tight_layout()
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()
    plt.close()

    # Print basic statistics
    print('\nStatistics for numerical columns:')
    for col in numerical_cols:
        print(f'\n{col}:')
        print(df[col].describe())

    # Combined plot for categorical columns
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(categorical_cols, 1):
        plt.subplot(2, 4, i)  # 2 rows, 4 columns for 7 categorical variables
        sns.countplot(data=df, x=col, order=df[col].value_counts().index, palette='viridis')
        plt.title(col, fontsize=12)
        plt.xlabel('')
        plt.ylabel('Frequency' if i % 4 == 1 else '', fontsize=10)
        plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    plt.close()

    # Print value counts
    print('\nValue counts for categorical columns:')
    for col in categorical_cols:
        print(f'\n{col}:')
        print(df[col].value_counts())


univariate_analysis(df)

### `Bi-Variate Analysis`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def perform_bivariate_analysis(df):
    # Set style for better visualization
    plt.style.use('seaborn-v0_8')
    
    # 1. Continuous vs Continuous: Mileage vs Procurement Cost
    def plot_continuous_bivariate():
        # Create figure for scatter plot
        plt.figure(figsize=(8, 6))
        
        # Original Scatter Plot
        plt.scatter(df['mileage'], df['procurement_cost'], alpha=0.5)
        plt.title('Mileage vs Procurement Cost')
        plt.xlabel('Mileage')
        plt.ylabel('Procurement Cost')
        
        # Calculate Pearson correlation
        correlation = df['mileage'].corr(df['procurement_cost'])
        plt.text(0.05, 0.95, f'Pearson Correlation: {correlation:.2f}', 
                transform=plt.gca().transAxes)
        
        plt.tight_layout()
        plt.savefig('mileage_vs_procurement_cost.png')
        plt.show()  # Display the plot
        plt.close()
    
    # 2. Continuous vs Categorical: Box Plots in a Single Figure
    def plot_categorical_bivariate():
        categorical_columns = ['brand', 'fuel_type', 'transmission_type', 
                            'prev_owner_count', 'car_type', 'price_segment']
        
        # Create a grid of subplots (2 rows, 3 columns)
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()  # Flatten to easily iterate over
        
        # Define a color palette for different categories
        palette = sns.color_palette("husl", n_colors=10)  # Use 'husl' palette, adjust n_colors if needed
        
        for idx, col in enumerate(categorical_columns):
            # Limit to top 10 categories to avoid overcrowding
            top_categories = df[col].value_counts().index[:10]
            filtered_df = df[df[col].isin(top_categories)]
            
            # Create box plot with different colors for each category
            sns.boxplot(x=col, y='procurement_cost', data=filtered_df, 
                       ax=axes[idx], palette=palette)
            axes[idx].set_title(f'Procurement Cost by {col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Procurement Cost')
            axes[idx].tick_params(axis='x', rotation=45)
        
        # Adjust layout to prevent overlap
        plt.tight_layout()
        plt.savefig('boxplots_all_categorical.png')
        plt.show()  # Display the plot
        plt.close()
    
    # Execute both analyses
    plot_continuous_bivariate()
    plot_categorical_bivariate()

# Example usage:
perform_bivariate_analysis(df)

### Data Set for `Smoothing` and `Optimization`

In [ ]:
df.describe()

#### Binning

In [ ]:
import numpy as np
import pandas as pd

# Copy the dataframe
df_o = df.copy()

# Filter out negative mileage values (assumed invalid)
df_o = df_o[df_o['mileage'] >= 0]

# Calculate number of bins using square root method: k = sqrt(n)
n_rows = len(df_o)
sqrt_bins = int(np.sqrt(n_rows))

# Create equal-width bins for mileage column
df_o['mileage_bins'] = pd.cut(df_o['mileage'], bins=sqrt_bins, include_lowest=True)

# Calculate midpoints for each bin
df_o['mileage_o'] = df_o['mileage_bins'].apply(lambda x: x.mid)

# Create bin range column
df_o['mileage_bin_range'] = df_o['mileage_bins'].apply(lambda x: f'[{x.left:.2f}, {x.right:.2f}]')

# Calculate median procurement_cost for each bin
median_costs = df_o.groupby('mileage_bins')['procurement_cost'].median()

# Map median costs to a new column based on mileage_bins
df_o['procurement_cost_o'] = df_o['mileage_bins'].map(median_costs)

# Display first few rows
print(f"Number of bins (Square root method): {sqrt_bins}")
display(df_o[['mileage', 'mileage_bins', 'mileage_o', 'mileage_bin_range', 'procurement_cost', 'procurement_cost_o']].head())

In [ ]:
# Select relevant columns and drop duplicates based on 'mileage_o'
df_o = df_o[['mileage_o', 'procurement_cost_o', 'brand_new_on_road_price']].copy()

# Drop rows with the same 'mileage_o'
df_o = df_o.drop_duplicates(subset='mileage_o')

# Display the shape of the dataframe
df_o.shape

#### Procurement Cost

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

# Extract data for plotting
x = df_o['mileage_o']
y = df_o['procurement_cost_o']

# Calculate Pearson correlation
corr, _ = pearsonr(x, y)
print(f"Pearson correlation coefficient: {corr:.3f}")

# Create scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, color='blue', alpha=0.5, label='Data points')

# Add linear regression trend line
z = np.polyfit(x, y, 3)  # Fit a 1st-degree polynomial (linear)
p = np.poly1d(z)  # Create polynomial object

# Calculate R²
y_pred = p(x)
ss_tot = np.sum((y - np.mean(y))**2)
ss_res = np.sum((y - y_pred)**2)
r2 = 1 - (ss_res / ss_tot)

# Plot trend line with R² and correlation in legend
plt.plot(x, p(x), color='red', linestyle='--', label=f'Trend line\nR²={r2:.3f}, corr={corr:.3f}')

# Add labels and title
plt.xlabel('Mileage (midpoint)')
plt.ylabel('Procurement Cost (median)')
plt.title('Procurement Cost vs Mileage with Correlation')
plt.legend()
plt.grid(True)

# Save plot
plt.savefig('procurement_cost_vs_mileage.png')

# Show plot
plt.show()

In [ ]:
df_o.head()

#### Remanufacturing Cost

In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import quad

def calculate_remanufacturing_cost(df, mileage_col, procurement_col, Cf=1000, a=0.25, b=8, h0=0, scale_factor=0.33):
    
    # Integration function
    def integrand(Kd):
        return (a * b) * (a * Kd)**(b - 1) + h0

    def compute_Cr(mileage):
        integral_value, _ = quad(integrand, 0, mileage)
        return integral_value

    # Compute Cr
    df["Cr"] = df[mileage_col].apply(compute_Cr)
    df["Cr"] = pd.to_numeric(df["Cr"], errors='coerce')
    
    # Scaling
    Cr_min = df["Cr"].min()
    Cr_max = df["Cr"].max()
    proc_max = df[procurement_col].max()
    
    scale = Cr_max / (scale_factor * proc_max)
    df["remanufacturing_cost"] = (df["Cr"] / scale) + Cf

    return df


In [ ]:
df_o = calculate_remanufacturing_cost(df_o, mileage_col="mileage_o", procurement_col="procurement_cost_o")

#### Total Cost

In [ ]:
df_o['Total_Cost'] = df_o['procurement_cost_o'] + df_o['remanufacturing_cost']
df_o.shape

In [ ]:
import matplotlib.pyplot as plt

def plot_costs_side_by_side(df, mileage_col="mileage_o", remanufacturing_col="remanufacturing_cost", 
                             procurement_col="procurement_cost_o", total_cost_col="Total_Cost"):
  
    # Create figure and 3 horizontal subplots
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))  # (rows, columns)

    # Plot 1: Remanufacturing Cost
    axs[0].plot(df[mileage_col], df[remanufacturing_col], color='blue', marker='o')
    axs[0].set_title('Remanufacturing Cost vs Mileage')
    axs[0].set_xlabel('Mileage')
    axs[0].set_ylabel('Remanufacturing Cost')
    axs[0].grid(True)

    # Plot 2: Procurement Cost
    axs[1].plot(df[mileage_col], df[procurement_col], color='green', marker='x')
    axs[1].set_title('Procurement Cost vs Mileage')
    axs[1].set_xlabel('Mileage')
    axs[1].set_ylabel('Procurement Cost')
    axs[1].grid(True)

    # Plot 3: Total Cost
    axs[2].plot(df[mileage_col], df[total_cost_col], color='red', marker='s')
    axs[2].set_title('Total Cost vs Mileage')
    axs[2].set_xlabel('Mileage')
    axs[2].set_ylabel('Total Cost')
    axs[2].grid(True)

    # Adjust layout
    plt.tight_layout()

    # Show plot
    plt.show()

# Example usage:
plot_costs_side_by_side(df_o)


#### Profit

In [ ]:
# Define selling price percentages
selling_percentages = [0.6, 0.7, 0.8]

# Loop through each percentage and calculate profit
for perc in selling_percentages:
    selling_price_col = f"selling_price_{int(perc*100)}"  # e.g., selling_price_60
    profit_col = f"profit_{int(perc*100)}"                # e.g., profit_60
    
    # Selling Price
    df_o[selling_price_col] = perc * df_o["brand_new_on_road_price"]
    
    # Profit = Selling price - Remanufacturing cost
    df_o[profit_col] = df_o[selling_price_col] - df_o["Total_Cost"]

# Check the results
df_o.shape


In [ ]:
import matplotlib.pyplot as plt

# Create side-by-side subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot Profit at 60% Selling Price
axes[0].plot(df_o["mileage_o"], df_o["profit_60"], color='b', label="Profit at 60%")
axes[0].axhline(0, color='black',linewidth=1)  # Zero line
axes[0].set_title("Profit at 60% Selling Price")
axes[0].set_xlabel("Mileage (Km)")
axes[0].set_ylabel("Profit")
axes[0].legend()
axes[0].grid(True)

# Plot Profit at 70% Selling Price
axes[1].plot(df_o["mileage_o"], df_o["profit_70"], color='g', label="Profit at 70%")
axes[1].axhline(0, color='black',linewidth=1)  # Zero line
axes[1].set_title("Profit at 70% Selling Price")
axes[1].set_xlabel("Mileage (Km)")
axes[1].set_ylabel("Profit")
axes[1].legend()
axes[1].grid(True)

# Plot Profit at 80% Selling Price
axes[2].plot(df_o["mileage_o"], df_o["profit_80"], color='r', label="Profit at 80%")
axes[2].axhline(0, color='black',linewidth=1)  # Zero line
axes[2].set_title("Profit at 80% Selling Price")
axes[2].set_xlabel("Mileage (Km)")
axes[2].set_ylabel("Profit")
axes[2].legend()
axes[2].grid(True)

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()

#### Auto Correlation

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.stattools import pacf

# Create a figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot PACF for Profit at 60% Selling Price
plot_pacf(df_o["profit_60"], ax=axes[0], lags=40, method='ywm')
axes[0].set_title("PACF of Profit at 60% Selling Price")
axes[0].set_xlabel("Lags")
axes[0].set_ylabel("Partial Autocorrelation")

# Plot PACF for Profit at 70% Selling Price
plot_pacf(df_o["profit_70"], ax=axes[1], lags=40, method='ywm')
axes[1].set_title("PACF of Profit at 70% Selling Price")
axes[1].set_xlabel("Lags")
axes[1].set_ylabel("Partial Autocorrelation")

# Plot PACF for Profit at 80% Selling Price
plot_pacf(df_o["profit_80"], ax=axes[2], lags=40, method='ywm')
axes[2].set_title("PACF of Profit at 80% Selling Price")
axes[2].set_xlabel("Lags")
axes[2].set_ylabel("Partial Autocorrelation")

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()


#### Modelling Profits in Each Case

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score

# Function to model trend using LassoCV
def model_trend(X, y, degree=5):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_poly)
    
    lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso.fit(X_scaled, y)
    
    y_trend = lasso.predict(X_scaled)
    r2 = r2_score(y, y_trend)
    
    return y_trend, r2, lasso, scaler, poly

# Function for Exponential Moving Average (EMA)
def ema(series, span=14):
    return series.ewm(span=span, adjust=False).mean()

# Main function to analyze trend and apply EMA
# Main function to analyze trend and apply EMA
def analyze_trend_with_ema(df, target_cols=['profit_60', 'profit_70', 'profit_80'], lags=2, degree=3, ema_span=14):
    # Create a subplot for each target column
    fig, axes = plt.subplots(1, len(target_cols), figsize=(18, 7))
    
    r2_scores = {}  # Initialize a dictionary to store R² scores for each target column
    
    for i, target_col in enumerate(target_cols):
        # Create lagged features (lags 1 and 2)
        df_lagged = df[['mileage_o', target_col]].copy()
        for lag in range(1, lags + 1):
            df_lagged[f'{target_col}_lag_{lag}'] = df_lagged[target_col].shift(lag)
        df_lagged = df_lagged.dropna()

        X = df_lagged[['mileage_o'] + [f'{target_col}_lag_{i}' for i in range(1, lags + 1)]].values
        y = df_lagged[target_col].values
        mileage = df_lagged['mileage_o'].values.astype(float)
        
        # Step 1: Model the trend using Lasso
        y_trend, r2_trend, lasso_model, scaler, poly = model_trend(X, y, degree)
        
        # Step 2: Apply EMA to smooth the trend
        y_trend_ema = ema(pd.Series(y_trend), span=ema_span)
        
        # Drop NaNs caused by smoothing
        valid_idx = (~np.isnan(y_trend)) & (~np.isnan(y_trend_ema))
        
        # Apply valid_idx to everything
        y = y[valid_idx]
        mileage = mileage[valid_idx]
        y_trend = y_trend[valid_idx]
        y_trend_ema = y_trend_ema[valid_idx]

        # Plot each target column
        axes[i].scatter(mileage, y, color='lightgray', alpha=0.6, label='Original Data')
        axes[i].plot(mileage, y_trend, color='red', label=f'Trend (R²={r2_trend:.4f})')
        axes[i].plot(mileage, y_trend_ema, color='orange', label=f'Trend + EMA')
        axes[i].set_xlabel('Mileage')
        axes[i].set_ylabel(target_col)
        axes[i].set_title(f'Trend + EMA for {target_col}')
        axes[i].legend()
        axes[i].grid(True)

        # Store R² score for the current target column
        r2_scores[target_col] = r2_trend
    
    plt.tight_layout()
    plt.show()

    return r2_scores

# Example usage
# df_o = pd.read_csv('your_data.csv')
r2_scores = analyze_trend_with_ema(df_o, target_cols=['profit_60', 'profit_70', 'profit_80'])
print(r2_scores)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import fminbound

# --- 1) helper: exponential moving average ---
def ema(series: pd.Series, span: int) -> np.ndarray:
    return series.ewm(span=span, adjust=False).mean().to_numpy()

# --- 2) helper: find the global maximum of the EMA curve ---
def find_maxima(
    mileage: np.ndarray,
    y_ema:     np.ndarray,
    refine:   bool = False
) -> tuple[float, float]:
    """
    If refine=False, returns the argmax sample.
    If refine=True, builds a cubic interpolator + fminbound for sub-sample precision.
    """
    if not refine:
        idx = np.argmax(y_ema)
        return float(mileage[idx]), float(y_ema[idx])
    # build cubic interpolator and maximize via fminbound on -f(x)
    f = interp1d(mileage, y_ema, kind='cubic')
    opt_m = fminbound(lambda x: -f(x), mileage.min(), mileage.max())
    return float(opt_m), float(f(opt_m))

# --- 3) helper: find *all* break-even crossings at threshold=0 ---
def find_break_even(
    mileage: np.ndarray,
    y_ema:     np.ndarray,
    threshold: float = 0
) -> list[tuple[float,float]]:
    """
    Returns a list of (mileage, threshold) for every crossing of y_ema through 'threshold'.
    Uses linear interpolation between samples to estimate the exact crossing X.
    """
    sig = np.sign(y_ema - threshold)
    crossings = np.where(np.diff(sig) != 0)[0]
    beps = []
    for idx in crossings:
        x0, x1 = mileage[idx],   mileage[idx+1]
        y0, y1 = y_ema[idx]-threshold, y_ema[idx+1]-threshold
        # linear interp to zero: x = x0 − y0*(x1−x0)/(y1−y0)
        x_be = x0 - y0 * (x1 - x0) / (y1 - y0)
        beps.append((float(x_be), float(threshold)))
    return beps

# --- 4) the main analysis + plotting function ---
def analyze_trend_with_ema(
    df:           pd.DataFrame,
    target_cols:  list[str] = ['profit_60','profit_70','profit_80'],
    lags:         int    = 2,
    degree:       int    = 3,
    ema_span:     int    = 14,
    refine_max:   bool   = False
) -> tuple[dict, dict, dict]:
    """
    For each target in target_cols:
      • Builds lagged DataFrame
      • Fits model_trend(...) → (y_trend, r2, ...)
      • Smooths y_trend with EMA
      • Finds maxima + all BEPs
      • Plots data, trend, EMA, and annotates points
    Returns three dicts keyed by target_col: r2_scores, maxima, break_even_points.
    """
    # --- You must have defined `model_trend(X,y,degree)` elsewhere; it returns (y_trend, r2, model, scaler, poly) ---
    r2_scores = {}
    maxima      = {}
    beps_all    = {}

    fig, axes = plt.subplots(1, len(target_cols), figsize=(18,6))

    for ax, target in zip(axes, target_cols):
        # 1) prepare lagged data
        df2 = df[['mileage_o', target]].copy()
        for lag in range(1, lags+1):
            df2[f'{target}_lag_{lag}'] = df2[target].shift(lag)
        df2 = df2.dropna()

        X = df2[['mileage_o'] + [f'{target}_lag_{i}' for i in range(1, lags+1)]].values
        y = df2[target].values
        mileage = df2['mileage_o'].values.astype(float)

        # 2) fit trend model (you supply this)
        y_trend, r2_trend, _, _, _ = model_trend(X, y, degree)

        # 3) smooth with EMA
        y_trend_ema = ema(pd.Series(y_trend), span=ema_span)

        # 4) find peaks & crossings
        max_m, max_v = find_maxima(mileage, y_trend_ema, refine=refine_max)
        beps         = find_break_even(mileage, y_trend_ema, threshold=0)

        # 5) plot
        ax.scatter(mileage, y,         color='lightgray', alpha=0.5, label='Data')
        ax.plot(   mileage, y_trend,   color='red',    label=f'Trend (R²={r2_trend:.3f})')
        ax.plot(   mileage, y_trend_ema, color='orange', label='Trend + EMA')

        # annotate max
        ax.scatter([max_m], [max_v], color='blue', zorder=5)
        ax.annotate(f"Max\n{int(max_m)}km, {max_v:,.0f}",
                    xy=(max_m,max_v), xytext=(0,20),
                    textcoords='offset points', ha='center',
                    arrowprops=dict(arrowstyle='->'))

        # annotate each BEP
        for i,(m_be, v_be) in enumerate(beps):
            ax.scatter([m_be],[v_be], color='green', zorder=5)
            ax.annotate(f"BEP{i+1}\n{int(m_be)}km",
                        xy=(m_be,v_be), xytext=(5,-15),
                        textcoords='offset points', ha='left')

        ax.set_xlabel('Mileage')
        ax.set_ylabel(target)
        ax.set_title(target)
        ax.legend()
        ax.grid(True)

        # store results
        r2_scores[target] = r2_trend
        maxima[target]     = (max_m, max_v)
        beps_all[target]   = beps

    plt.tight_layout()
    plt.show()

    return r2_scores, maxima, beps_all

# --- 5) Example usage ---
if __name__ == "__main__":
    # load your data (must have columns: mileage_o, profit_60, profit_70, profit_80)
    # df_o = pd.read_csv("your_data.csv")

    # run with sub-sample refine for the max
    r2_scores, maxima, break_even_pts = analyze_trend_with_ema(
        df_o,
        target_cols=['profit_60','profit_70','profit_80'],
        lags=2,
        degree=3,
        ema_span=14,
        refine_max=True
    )

    print("R² Scores:\n", r2_scores)
    print("Maxima:\n", maxima)
    print("All BEPs:\n", break_even_pts)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import fminbound

# --- 1) helper: exponential moving average ---
def ema(series: pd.Series, span: int) -> np.ndarray:
    return series.ewm(span=span, adjust=False).mean().to_numpy()

# --- 2) helper: find the global maximum of the EMA curve ---
def find_maxima(mileage: np.ndarray, y_ema: np.ndarray, refine: bool = False) -> tuple[float, float]:
    if not refine:
        idx = np.argmax(y_ema)
        return float(mileage[idx]), float(y_ema[idx])
    f = interp1d(mileage, y_ema, kind='cubic')
    opt_m = fminbound(lambda x: -f(x), mileage.min(), mileage.max())
    return float(opt_m), float(f(opt_m))

# --- 3) helper: find all break-even points ---
def find_break_even(mileage: np.ndarray, y_ema: np.ndarray, threshold: float = 0) -> list[tuple[float, float]]:
    sig = np.sign(y_ema - threshold)
    crossings = np.where(np.diff(sig) != 0)[0]
    beps = []
    for idx in crossings:
        x0, x1 = mileage[idx], mileage[idx+1]
        y0, y1 = y_ema[idx] - threshold, y_ema[idx+1] - threshold
        x_be = x0 - y0 * (x1 - x0) / (y1 - y0)
        beps.append((float(x_be), float(threshold)))
    return beps

# --- 4) the main plotting function ---
def analyze_trend_with_ema(
    df: pd.DataFrame,
    target_cols: list[str] = ['profit_60', 'profit_70', 'profit_80'],
    lags: int = 2,
    degree: int = 3,
    ema_span: int = 14,
    refine_max: bool = False
) -> tuple[dict, dict, dict]:

    r2_scores = {}
    maxima = {}
    beps_all = {}

    plot_titles = {
        'profit_60': 'Profit with 40% discount on Brand New Car',
        'profit_70': 'Profit with 30% discount on Brand New Car',
        'profit_80': 'Profit with 20% discount on Brand New Car'
    }

    for target in target_cols:
        fig, ax = plt.subplots(figsize=(10, 6))

        df2 = df[['mileage_o', target]].copy()
        for lag in range(1, lags + 1):
            df2[f'{target}_lag_{lag}'] = df2[target].shift(lag)
        df2 = df2.dropna()

        X = df2[['mileage_o'] + [f'{target}_lag_{i}' for i in range(1, lags + 1)]].values
        y = df2[target].values
        mileage = df2['mileage_o'].values.astype(float)

        # Your model_trend must be defined separately
        y_trend, r2_trend, _, _, _ = model_trend(X, y, degree)
        y_trend_ema = ema(pd.Series(y_trend), span=ema_span)

        max_m, max_v = find_maxima(mileage, y_trend_ema, refine=refine_max)
        beps = find_break_even(mileage, y_trend_ema)

        # --- plot ---
        ax.scatter(mileage, y, color='lightgray', alpha=0.5, label='Data')
        ax.plot(mileage, y_trend, color='red', label=f'Trend (R²={r2_trend:.3f})')
        ax.plot(mileage, y_trend_ema, color='orange', label='Trend + EMA')

        # Max annotation
        ax.scatter([max_m], [max_v], color='blue', zorder=5)
        ax.annotate(f"Max\n{int(max_m)}km, Rs.{max_v:,.0f}",
                    xy=(max_m, max_v), xytext=(0, 20),
                    textcoords='offset points', ha='center',
                    fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=0.5),
                    arrowprops=dict(arrowstyle='->', lw=0.5))

        # Annotate only last 2 BEPs
        beps = sorted(beps, key=lambda x: x[0])  # sort by mileage
        last_beps = beps[-2:] if len(beps) >= 2 else beps

        for i, (m_be, v_be) in enumerate(last_beps):
            ax.scatter([m_be], [v_be], color='green', zorder=5)
            ax.annotate(f"BEP{i+1}\n{int(m_be)}km",
                        xy=(m_be, v_be), xytext=(5, -15 - 15*i),
                        textcoords='offset points', ha='left',
                        fontsize=8,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=0.5),
                        arrowprops=dict(arrowstyle='->', lw=0.5))

        ax.set_xlabel('Mileage')
        ax.set_ylabel('Profit (Rs.)')
        ax.set_title(plot_titles.get(target, target))
        ax.legend()
        ax.grid(True)

        plt.tight_layout()
        plt.show()

        r2_scores[target] = r2_trend
        maxima[target] = (max_m, max_v)
        beps_all[target] = beps

    return r2_scores, maxima, beps_all

# --- 5) Example usage ---
if __name__ == "__main__":
    # Load your data
    # df_o = pd.read_csv('your_data.csv')

    r2_scores, maxima, break_even_pts = analyze_trend_with_ema(
        df_o,
        target_cols=['profit_60', 'profit_70', 'profit_80'],
        lags=2,
        degree=3,
        ema_span=14,
        refine_max=True
    )

    print("R² Scores:\n", r2_scores)
    print("Maxima:\n", maxima)
    print("All BEPs:\n", break_even_pts)


### `Sensitivity Analysis`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.interpolate import interp1d
from scipy.optimize import fminbound
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score

# Define calculate_remanufacturing_cost
def calculate_remanufacturing_cost(df, mileage_col, procurement_col, Cf=1000, a=0.25, b=8, h0=0, scale_factor=0.33):
    def integrand(Kd):
        return (a * b) * (a * Kd)**(b - 1) + h0

    def compute_Cr(mileage):
        integral_value, _ = quad(integrand, 0, mileage)
        return integral_value

    df["Cr"] = df[mileage_col].apply(compute_Cr)
    df["Cr"] = pd.to_numeric(df["Cr"], errors='coerce')
    
    Cr_min = df["Cr"].min()
    Cr_max = df["Cr"].max()
    proc_max = df[procurement_col].max()
    
    scale = Cr_max / (scale_factor * proc_max)
    df["remanufacturing_cost"] = (df["Cr"] / scale) + Cf
    return df

# Define model_trend
def model_trend(X, y, degree=5):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_poly)
    
    lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso.fit(X_scaled, y)
    
    y_trend = lasso.predict(X_scaled)
    r2 = r2_score(y, y_trend)
    return y_trend, r2, lasso, scaler, poly

# Define EMA helper
def ema(series: pd.Series, span: int) -> np.ndarray:
    return series.ewm(span=span, adjust=False).mean().to_numpy()

# Define find_maxima helper
def find_maxima(mileage: np.ndarray, y_ema: np.ndarray, refine: bool = False) -> tuple[float, float]:
    if not refine:
        idx = np.argmax(y_ema)
        return float(mileage[idx]), float(y_ema[idx])
    f = interp1d(mileage, y_ema, kind='cubic')
    opt_m = fminbound(lambda x: -f(x), mileage.min(), mileage.max())
    return float(opt_m), float(f(opt_m))

# Define find_break_even helper
def find_break_even(mileage: np.ndarray, y_ema: np.ndarray, threshold: float = 0) -> list[tuple[float, float]]:
    sig = np.sign(y_ema - threshold)
    crossings = np.where(np.diff(sig) != 0)[0]
    beps = []
    for idx in crossings:
        x0, x1 = mileage[idx], mileage[idx + 1]
        y0, y1 = y_ema[idx] - threshold, y_ema[idx + 1] - threshold
        x_be = x0 - y0 * (x1 - x0) / (y1 - y0)
        beps.append((float(x_be), float(threshold)))
    return beps

# Define analyze_trend_with_ema
def analyze_trend_with_ema(
    df: pd.DataFrame,
    target_cols: list[str] = ['profit_60', 'profit_70', 'profit_80'],
    lags: int = 2,
    degree: int = 3,
    ema_span: int = 14,
    refine_max: bool = True,
    plot: bool = False
) -> tuple[dict, dict, dict]:
    r2_scores = {}
    maxima = {}
    beps_all = {}

    for target in target_cols:
        df2 = df[['mileage_o', target]].copy()
        for lag in range(1, lags + 1):
            df2[f'{target}_lag_{lag}'] = df2[target].shift(lag)
        df2 = df2.dropna()

        X = df2[['mileage_o'] + [f'{target}_lag_{i}' for i in range(1, lags + 1)]].values
        y = df2[target].values
        mileage = df2['mileage_o'].values.astype(float)

        y_trend, r2_trend, _, _, _ = model_trend(X, y, degree)
        y_trend_ema = ema(pd.Series(y_trend), span=ema_span)

        max_m, max_v = find_maxima(mileage, y_trend_ema, refine=refine_max)
        beps = find_break_even(mileage, y_trend_ema)
        beps = sorted(beps, key=lambda x: x[0])  # Sort by mileage
        last_beps = beps[-2:] if len(beps) >= 2 else beps + [(np.nan, np.nan)] * (2 - len(beps))

        r2_scores[target] = r2_trend
        maxima[target] = (max_m, max_v)
        beps_all[target] = last_beps

        if plot:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(mileage, y_trend_ema, label=f'{target} Trend + EMA')
            ax.scatter([max_m], [max_v], color='blue', label='Maximum')
            for i, (m_be, _) in enumerate(last_beps):
                if not np.isnan(m_be):
                    ax.scatter([m_be], [0], color='green', label=f'BEP{i+1}')
            ax.set_xlabel('Mileage (km)')
            ax.set_ylabel('Profit (Rs.)')
            ax.set_title(f'{target} Analysis')
            ax.legend()
            ax.grid(True)
            plt.show()

    return r2_scores, maxima, beps_all

# Define sensitivity_analysis with percentage change calculations
def sensitivity_analysis(df, param_name, param_values, default_params):
    target_cols = ['profit_60', 'profit_70', 'profit_80']
    
    # Calculate baseline values at default parameters
    df_default = calculate_remanufacturing_cost(df.copy(), "mileage_o", "procurement_cost_o", **default_params)
    df_default['Total_Cost'] = df_default['procurement_cost_o'] + df_default['remanufacturing_cost']
    
    for perc in [0.6, 0.7, 0.8]:
        selling_price_col = f"selling_price_{int(perc*100)}"
        profit_col = f"profit_{int(perc*100)}"
        df_default[selling_price_col] = perc * df_default["brand_new_on_road_price"]
        df_default[profit_col] = df_default[selling_price_col] - df_default['Total_Cost']
    
    _, maxima_default, beps_default = analyze_trend_with_ema(df_default, target_cols, plot=False)
    
    # Initialize dictionaries to store percentage changes
    opt_mileage_pct = {target: [] for target in target_cols}
    max_profit_pct = {target: [] for target in target_cols}
    bep1_pct = {target: [] for target in target_cols}
    bep2_pct = {target: [] for target in target_cols}
    param_pct_change = []

    default_value = default_params[param_name]
    
    for val in param_values:
        params = default_params.copy()
        params[param_name] = val
        df_temp = calculate_remanufacturing_cost(df.copy(), "mileage_o", "procurement_cost_o", **params)
        df_temp['Total_Cost'] = df_temp['procurement_cost_o'] + df_temp['remanufacturing_cost']
        
        for perc in [0.6, 0.7, 0.8]:
            selling_price_col = f"selling_price_{int(perc*100)}"
            profit_col = f"profit_{int(perc*100)}"
            df_temp[selling_price_col] = perc * df_temp["brand_new_on_road_price"]
            df_temp[profit_col] = df_temp[selling_price_col] - df_temp['Total_Cost']
        
        _, maxima, beps_all = analyze_trend_with_ema(df_temp, target_cols, plot=False)
        
        # Calculate percentage change in parameter
        param_change = (val - default_value) / default_value * 100
        param_pct_change.append(param_change)
        
        for target in target_cols:
            # Percentage change in optimum mileage
            opt_mileage_change = (maxima[target][0] - maxima_default[target][0]) / maxima_default[target][0] * 100
            opt_mileage_pct[target].append(opt_mileage_change)
            
            # Percentage change in maximum profit
            max_profit_change = (maxima[target][1] - maxima_default[target][1]) / maxima_default[target][1] * 100
            max_profit_pct[target].append(max_profit_change)
            
            # Percentage change in BEP1
            bep1_default = beps_default[target][0][0] if len(beps_default[target]) > 0 else np.nan
            bep1_val = beps_all[target][0][0] if len(beps_all[target]) > 0 else np.nan
            if not np.isnan(bep1_default) and not np.isnan(bep1_val):
                bep1_change = (bep1_val - bep1_default) / bep1_default * 100
            else:
                bep1_change = np.nan
            bep1_pct[target].append(bep1_change)
            
            # Percentage change in BEP2
            bep2_default = beps_default[target][1][0] if len(beps_default[target]) > 1 else np.nan
            bep2_val = beps_all[target][1][0] if len(beps_all[target]) > 1 else np.nan
            if not np.isnan(bep2_default) and not np.isnan(bep2_val):
                bep2_change = (bep2_val - bep2_default) / bep2_default * 100
            else:
                bep2_change = np.nan
            bep2_pct[target].append(bep2_change)

    return param_pct_change, opt_mileage_pct, max_profit_pct, bep1_pct, bep2_pct

# Define plot_sensitivity with percentage changes
def plot_sensitivity(param_name, param_pct_change, opt_mileage_pct, max_profit_pct, bep1_pct, bep2_pct, target_cols):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for target in target_cols:
        axes[0].plot(param_pct_change, opt_mileage_pct[target], label=target)
        axes[1].plot(param_pct_change, max_profit_pct[target], label=target)
        axes[2].plot(param_pct_change, bep1_pct[target], label=f'{target} BEP1')
        axes[3].plot(param_pct_change, bep2_pct[target], label=f'{target} BEP2')

    titles = ['% Change in Optimum Mileage', '% Change in Maximum Profit', '% Change in BEP1', '% Change in BEP2']
    ylabels = ['% Change', '% Change', '% Change', '% Change']
    for i, ax in enumerate(axes):
        ax.set_xlabel(f'% Change in {param_name}')
        ax.set_ylabel(ylabels[i])
        ax.set_title(f'{titles[i]} vs % Change in {param_name}')
        ax.legend()
        ax.grid(True)

    plt.tight_layout()
    plt.show()

# Main execution
if __name__ == "__main__":

    default_params = {'a': 0.25, 'b': 8, 'h0': 0, 'scale_factor': 0.33, 'Cf': 1000}
    param_ranges = {
        'a': np.arange(0.1, 0.51, 0.05),
        'b': np.arange(5, 11, 1),
        'h0': np.arange(0, 101, 20),
        'scale_factor': np.arange(0.1, 0.51, 0.05)
    }

    # Use df_o for the analysis
    for param_name, param_values in param_ranges.items():
        param_pct_change, opt_mileage_pct, max_profit_pct, bep1_pct, bep2_pct = sensitivity_analysis(df_o, param_name, param_values, default_params)
        plot_sensitivity(param_name, param_pct_change, opt_mileage_pct, max_profit_pct, bep1_pct, bep2_pct, ['profit_60', 'profit_70', 'profit_80'])

    print("Sensitivity analysis completed. Plots displayed.")

### `Prediction Model`

In [ ]:
df_p = df.copy()

In [ ]:
df_p.info()

### Feature Engineering

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Load your data (if it's already loaded as df_p, skip this)
# df_p = pd.read_csv('your_file.csv')

# Quick check
df_p.head()


#### Encoding Categorical

In [ ]:
# Checking unique values in categorical columns
categorical_columns = ['brand', 'model', 'fuel_type', 'transmission_type', 'prev_owner_count', 'car_type', 'price_segment']

for col in categorical_columns:
    print(f"{col} : {df_p[col].nunique()} unique values")


In [ ]:
df_p = pd.get_dummies(df_p, columns=categorical_columns, drop_first=True)
df_p.head()


In [ ]:
df_p.info()

#### Linear Regression

In [ ]:
# Imports for model training
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Define target and features
target = 'procurement_cost'
features = df_p.drop(columns=[target])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, df_p[target], test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train['mileage'] = scaler.fit_transform(X_train[['mileage']])
X_val['mileage'] = scaler.transform(X_val[['mileage']])

# Initialize the model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

# R^2 scores
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# Plot procurement_cost vs mileage for validation data only
plt.figure(figsize=(10, 6))

# Plot for actual validation data
sns.scatterplot(x=X_val['mileage'], y=y_val, label='Actual Validation Data', color='red')

# Plot for predicted validation data
sns.scatterplot(x=X_val['mileage'], y=y_val_pred, label=f'Predicted Validation Data (R² = {val_r2:.2f})', color='green')

# Adding labels and title
plt.title(f'Procurement Cost vs Mileage (Validation) with R²\nTraining R² = {train_r2:.2f}, Validation R² = {val_r2:.2f}')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()

plt.show()

#### Decision Tree

In [ ]:
# Imports
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Define target and features
target = 'procurement_cost'
features = df_p.drop(columns=[target])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, df_p[target], test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train['mileage'] = scaler.fit_transform(X_train[['mileage']])
X_val['mileage'] = scaler.transform(X_val[['mileage']])
# Initialize the model
dt = DecisionTreeRegressor(random_state=42)

# Hyperparameter grid for grid search
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the grid search
grid_search.fit(X_train, y_train)

# Best model from grid search
best_dt = grid_search.best_estimator_

# Make predictions
y_train_pred = best_dt.predict(X_train)
y_val_pred = best_dt.predict(X_val)

# R² scores
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# Plot procurement_cost vs mileage for validation data only
plt.figure(figsize=(10, 6))

# Plot for actual validation data
sns.scatterplot(x=X_val['mileage'], y=y_val, label='Actual Validation Data', color='red')

# Plot for predicted validation data
sns.scatterplot(x=X_val['mileage'], y=y_val_pred, label=f'Predicted Validation Data (R² = {val_r2:.2f})', color='green')

# Adding labels and title
plt.title(f'Procurement Cost vs Mileage (Validation) with R²\nTraining R² = {train_r2:.2f}, Validation R² = {val_r2:.2f}')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()

plt.show()

# Print best parameters and score from grid search
print("Best Parameters from Grid Search:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)


#### Random Forest

In [ ]:
# Imports
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Define target and features
target = 'procurement_cost'
features = df_p.drop(columns=[target])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, df_p[target], test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train['mileage'] = scaler.fit_transform(X_train[['mileage']])
X_val['mileage'] = scaler.transform(X_val[['mileage']])
# Initialize the RandomForest model
rf = RandomForestRegressor(random_state=42)

# Hyperparameter grid for grid search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'bootstrap': [True, False]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the grid search
grid_search.fit(X_train, y_train)

# Best model from grid search
best_rf = grid_search.best_estimator_

# Make predictions
y_train_pred = best_rf.predict(X_train)
y_val_pred = best_rf.predict(X_val)

# R² scores
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# Plot procurement_cost vs mileage for validation data only
plt.figure(figsize=(10, 6))

# Plot for actual validation data
sns.scatterplot(x=X_val['mileage'], y=y_val, label='Actual Validation Data', color='red')

# Plot for predicted validation data
sns.scatterplot(x=X_val['mileage'], y=y_val_pred, label=f'Predicted Validation Data (R² = {val_r2:.2f})', color='green')

# Adding labels and title
plt.title(f'Procurement Cost vs Mileage (Validation) with R²\nTraining R² = {train_r2:.2f}, Validation R² = {val_r2:.2f}')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()

plt.show()

# Print best parameters and score from grid search
print("Best Parameters from Grid Search:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)


#### Gradient Boosting

In [ ]:
# Imports
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Define target and features
target = 'procurement_cost'
features = df_p.drop(columns=[target])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, df_p[target], test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train['mileage'] = scaler.fit_transform(X_train[['mileage']])
X_val['mileage'] = scaler.transform(X_val[['mileage']])
# Initialize the GradientBoosting model
gb = GradientBoostingRegressor(random_state=42)

# Hyperparameter grid for grid search
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.001, 0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=gb, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the grid search
grid_search.fit(X_train, y_train)

# Best model from grid search
best_gb = grid_search.best_estimator_

# Make predictions
y_train_pred = best_gb.predict(X_train)
y_val_pred = best_gb.predict(X_val)

# R² scores
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# Plot procurement_cost vs mileage for validation data only
plt.figure(figsize=(10, 6))

# Plot for actual validation data
sns.scatterplot(x=X_val['mileage'], y=y_val, label='Actual Validation Data', color='red')

# Plot for predicted validation data
sns.scatterplot(x=X_val['mileage'], y=y_val_pred, label=f'Predicted Validation Data (R² = {val_r2:.2f})', color='green')

# Adding labels and title
plt.title(f'Procurement Cost vs Mileage (Validation) with R²\nTraining R² = {train_r2:.2f}, Validation R² = {val_r2:.2f}')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()

plt.show()

# Print best parameters and score from grid search
print("Best Parameters from Grid Search:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)


#### XGBoost

In [ ]:
# pip install xgboost --quiet

In [ ]:
# Imports
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Define target and features
target = 'procurement_cost'
features = df_p.drop(columns=[target])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, df_p[target], test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train['mileage'] = scaler.fit_transform(X_train[['mileage']])
X_val['mileage'] = scaler.transform(X_val[['mileage']])
# Initialize the XGBoost model
xg_reg = xgb.XGBRegressor(random_state=42)

# Hyperparameter grid for grid search
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.001, 0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'min_child_weight': [1, 2, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=xg_reg, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the grid search
grid_search.fit(X_train, y_train)

# Best model from grid search
best_xg = grid_search.best_estimator_

# Make predictions
y_train_pred = best_xg.predict(X_train)
y_val_pred = best_xg.predict(X_val)

# R² scores
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# Plot procurement_cost vs mileage for validation data only
plt.figure(figsize=(10, 6))

# Plot for actual validation data
sns.scatterplot(x=X_val['mileage'], y=y_val, label='Actual Validation Data', color='red')

# Plot for predicted validation data
sns.scatterplot(x=X_val['mileage'], y=y_val_pred, label=f'Predicted Validation Data (R² = {val_r2:.2f})', color='green')

# Adding labels and title
plt.title(f'Procurement Cost vs Mileage (Validation) with R²\nTraining R² = {train_r2:.2f}, Validation R² = {val_r2:.2f}')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()

plt.show()

# Print best parameters and score from grid search
print("Best Parameters from Grid Search:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)


#### Light GBM

In [ ]:
# Imports
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Define target and features
target = 'procurement_cost'
features = df_p.drop(columns=[target])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, df_p[target], test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train['mileage'] = scaler.fit_transform(X_train[['mileage']])
X_val['mileage'] = scaler.transform(X_val[['mileage']])
# Initialize the LightGBM model
lgb_reg = lgb.LGBMRegressor(random_state=42)

# Hyperparameter grid for grid search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],
    'num_leaves': [31, 50, 100]
    }

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=lgb_reg, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the grid search
grid_search.fit(X_train, y_train)

# Best model from grid search
best_lgb = grid_search.best_estimator_

# Make predictions
y_train_pred = best_lgb.predict(X_train)
y_val_pred = best_lgb.predict(X_val)

# R² scores
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

# Plot procurement_cost vs mileage for validation data only
plt.figure(figsize=(10, 6))

# Plot for actual validation data
sns.scatterplot(x=X_val['mileage'], y=y_val, label='Actual Validation Data', color='red')

# Plot for predicted validation data
sns.scatterplot(x=X_val['mileage'], y=y_val_pred, label=f'Predicted Validation Data (R² = {val_r2:.2f})', color='green')

# Adding labels and title
plt.title(f'Procurement Cost vs Mileage (Validation) with R²\nTraining R² = {train_r2:.2f}, Validation R² = {val_r2:.2f}')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()

plt.show()

# Print best parameters and score from grid search
print("Best Parameters from Grid Search:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)


#### Stacking Regression

##### Random Forest + Gradient Boosting + XG Boost + Light GBM (Final Model)

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Assuming df_p is your dataframe with features and target
# Define target and features
target = 'procurement_cost'
features = df_p.drop(columns=[target])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, df_p[target], test_size=0.2, random_state=42)
X_train['mileage'] = scaler.fit_transform(X_train[['mileage']])
X_val['mileage'] = scaler.transform(X_val[['mileage']])

# Initialize base models with the best hyperparameters from grid search
rf_model = RandomForestRegressor(
    bootstrap=True, 
    max_depth=10, 
    min_samples_leaf=5, 
    min_samples_split=2, 
    n_estimators=100, 
    random_state=42
)

gb_model = GradientBoostingRegressor(
    learning_rate=0.1, 
    max_depth=5, 
    min_samples_leaf=5, 
    min_samples_split=2, 
    n_estimators=100, 
    random_state=42
)

xgb_model = xgb.XGBRegressor(
    colsample_bytree=0.6, 
    learning_rate=0.1, 
    max_depth=5, 
    min_child_weight=5, 
    n_estimators=100, 
    subsample=1.0, 
    random_state=42
)

# Create the final LightGBM model with the best parameters from grid search
lgbm_final = lgb.LGBMRegressor(
    max_depth=5, 
    n_estimators=100, 
    num_leaves=31, 
    random_state=42
)

# Stack the base models using StackingRegressor with LightGBM as the final estimator
base_models = [
    ('rf', rf_model),
    ('gb', gb_model),
    ('xgb', xgb_model)
]

stacked_model = StackingRegressor(
    estimators=base_models,
    final_estimator=lgbm_final,
    cv=5  # 5-fold cross-validation for stacking
)

# Train the stacked model
print("Training stacking model...")
stacked_model.fit(X_train, y_train)

# Evaluate individual base models
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    train_r2 = r2_score(y_train, y_train_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    
    print(f"{model_name} - Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}")
    print(f"{model_name} - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")
    print(f"{model_name} - Train MAE: {train_mae:.4f}, Val MAE: {val_mae:.4f}")
    print("-" * 50)
    
    return {
        'model_name': model_name,
        'train_r2': train_r2,
        'val_r2': val_r2,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_mae': train_mae,
        'val_mae': val_mae
    }

# Evaluate all models
print("Evaluating individual models...")
results = []
results.append(evaluate_model(rf_model, X_train, y_train, X_val, y_val, "Random Forest"))
results.append(evaluate_model(gb_model, X_train, y_train, X_val, y_val, "Gradient Boosting"))
results.append(evaluate_model(xgb_model, X_train, y_train, X_val, y_val, "XGBoost"))
results.append(evaluate_model(lgbm_final, X_train, y_train, X_val, y_val, "LightGBM"))

# Predict on train and validation data with stacked model
y_train_pred_stacked = stacked_model.predict(X_train)
y_val_pred_stacked = stacked_model.predict(X_val)

# Calculate metrics for stacked model
train_r2_stacked = r2_score(y_train, y_train_pred_stacked)
val_r2_stacked = r2_score(y_val, y_val_pred_stacked)
train_rmse_stacked = np.sqrt(mean_squared_error(y_train, y_train_pred_stacked))
val_rmse_stacked = np.sqrt(mean_squared_error(y_val, y_val_pred_stacked))
train_mae_stacked = mean_absolute_error(y_train, y_train_pred_stacked)
val_mae_stacked = mean_absolute_error(y_val, y_val_pred_stacked)

print("Stacked Model - Train R²: {:.4f}, Val R²: {:.4f}".format(train_r2_stacked, val_r2_stacked))
print("Stacked Model - Train RMSE: {:.4f}, Val RMSE: {:.4f}".format(train_rmse_stacked, val_rmse_stacked))
print("Stacked Model - Train MAE: {:.4f}, Val MAE: {:.4f}".format(train_mae_stacked, val_mae_stacked))

results.append({
    'model_name': 'Stacked Model',
    'train_r2': train_r2_stacked,
    'val_r2': val_r2_stacked,
    'train_rmse': train_rmse_stacked,
    'val_rmse': val_rmse_stacked,
    'train_mae': train_mae_stacked,
    'val_mae': val_mae_stacked
})

# Create a DataFrame for comparison
results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df)

# Visualizations

# 1. R² Comparison
plt.figure(figsize=(12, 6))
models = results_df['model_name']
x = np.arange(len(models))
width = 0.35

plt.bar(x - width/2, results_df['train_r2'], width, label='Training R²')
plt.bar(x + width/2, results_df['val_r2'], width, label='Validation R²')

plt.xlabel('Models')
plt.ylabel('R² Score')
plt.title('R² Comparison Across Models')
plt.xticks(x, models, rotation=45)
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 2. RMSE Comparison
plt.figure(figsize=(12, 6))
plt.bar(x - width/2, results_df['train_rmse'], width, label='Training RMSE')
plt.bar(x + width/2, results_df['val_rmse'], width, label='Validation RMSE')

plt.xlabel('Models')
plt.ylabel('RMSE')
plt.title('RMSE Comparison Across Models')
plt.xticks(x, models, rotation=45)
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 3. Actual vs Predicted Plot for Stacked Model
plt.figure(figsize=(10, 6))
plt.scatter(y_val, y_val_pred_stacked, alpha=0.5)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted (Stacked Model)')
plt.grid(True)
plt.tight_layout()
plt.show()

# 4. Residual Plot for Stacked Model
residuals = y_val - y_val_pred_stacked
plt.figure(figsize=(10, 6))
plt.scatter(y_val_pred_stacked, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot (Stacked Model)')
plt.grid(True)
plt.tight_layout()
plt.show()

# 5. Feature Importance (from LightGBM final estimator)
if hasattr(stacked_model.final_estimator_, 'feature_importances_'):
    # Get feature names (the base model predictions don't have names)
    feature_names = [f"Model_{i}" for i in range(len(base_models))]
    
    # Get feature importances
    importances = stacked_model.final_estimator_.feature_importances_
    
    # Create a DataFrame for visualization
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    })
    
    # Sort by importance
    importance_df = importance_df.sort_values('Importance', ascending=False)
    
    # Plot
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title('Feature Importance in Final LightGBM Model')
    plt.tight_layout()
    plt.show()

# Save the stacked model (optional)
import joblib
joblib.dump(stacked_model, 'stacked_model_with_lightgbm.pkl')
print("Model saved as 'stacked_model_with_lightgbm.pkl'")

In [ ]:
plt.figure(figsize=(10, 6))

# Plot for actual validation data
sns.scatterplot(x=X_val['mileage'], y=y_val, label='Actual Validation Data', color='red')

# Plot for predicted validation data
sns.scatterplot(x=X_val['mileage'], y=y_val_pred_stacked, label=f'Predicted Validation Data (R² = {val_r2_stacked:.3f})', color='green')

# Adding labels and title
plt.title(f'Procurement Cost vs Mileage (Validation) with R²\nTraining R² = {train_r2_stacked:.3f}, Validation R² = {val_r2_stacked:.3f}')
plt.xlabel('Mileage')
plt.ylabel('Procurement Cost')
plt.legend()

plt.show()